# RO change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import pandas as pd

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## specify args

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## specify T region
T_REGION = "3"

## should we use ML avg?
USE_ML_AVG = False

## use these...
varnames = [f"T_{T_REGION}", "h_hat"]

## Functions

In [ ]:
def var_by_quant(x):
    """compute variance by quantiles"""

    ## get bounds
    bounds = x.quantile(dim=["member", "time"], q=[0.25, 0.75]).rename(
        {"quantile": "q"}
    )

    ## find samples in each class
    in_class0 = x < bounds.isel(q=0)  ## extreme La Niña
    in_class1 = (bounds.isel(q=0) <= x) & (x <= bounds.isel(q=1))  ## neutral
    in_class2 = bounds.isel(q=1) < x  ## extreme El Niño

    ## put in single dataarray
    in_class = xr.concat(
        [in_class0, in_class1, in_class2],
        dim=pd.Index(np.arange(3), name="intensity"),
        coords="minimal",
        compat="override",
    )

    ## zero out NaNs
    # is_invalid = np.abs(x) > (x.std() * 7)
    # if is_invalid.to_dataarray().sum() > 0:

    #     ## count fraction of invalid
    #     frac = is_invalid.to_dataarray().mean().values.item()
    #     print(f"Warning: {frac:.1e} frac. large values")

    #     ## fill large values with NaN
    #     x = x.where(~is_invalid, other=np.nan)

    ## compute variance from each
    N = (~np.isnan(x)).sum(["member", "time"])
    x_var = 1 / N * (x.where(in_class) ** 2).sum(["time", "member"])

    return x_var


def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        # fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))
        fits.append(fit)

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims


def save(fig, fname, dpi=800, do_save=False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "june-updates")

    ## get fname
    fname = f"{fname}-{varnames[0]}-subtract_median_{REMOVE_MEDIAN}.pdf"

    if do_save:
        fig.savefig(save_dir / fname, dpi=dpi, format="pdf")

    return


def get_perturbed_NRO(params, nro_form_idx=None, nro_type="NROT_Lac"):
    """
    Fix values of R parameter set.
    if 'fix_others' is True, then other parameters are fixed to their
    initial value. Otherwise, given parameter is fixed to its initial value
    """

    ## initialize empty array to hold parameters
    params_new = copy.deepcopy(params)
    params_new[nro_type] = params_new[nro_type].transpose("year", "nro_form", ...)

    ## get numpy version of linear operator
    NRO_Lac = params_new[nro_type].values

    ## update parameter
    if nro_form_idx is None:
        NRO_Lac = NRO_Lac[:1]

    else:
        NRO_Lac[:, nro_form_idx] = NRO_Lac[:1, nro_form_idx]

    ## add back to parameters
    params_new[nro_type] = xr.ones_like(params_new[nro_type]) * NRO_Lac

    return params_new


def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})

    ## differentiate
    ddt_data = data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

    ## mult. by 12 to convert from 1/mo to 1/yr
    return 12 * ddt_data

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)
Th = Th.fillna(0)

#### Load ELI

In [ ]:
## Load ELI
eli_all = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
eli_forced, eli = src.utils.separate_forced(eli_all)

## add to data
Th = xr.merge([Th, eli])

Scale it

#### Merged Niño index

In [ ]:
## function to get merged nino3/34 regions
def get_nino_merged(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(190, 270))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Te(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(230, 280))
    return x.sel(idx).mean(["longitude", "latitude"])


def get_Tw(x):
    idx = dict(latitude=slice(-5, 5), longitude=slice(180, 230))
    return x.sel(idx).mean(["longitude", "latitude"])


## spatial data
forced, anom = src.utils.load_consolidated()

## compute
Th["T_m"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_nino_merged)[
    "sst"
]
Th["T_e"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Te)["sst"]
Th["T_w"] = src.utils.reconstruct_wrapper(anom[["sst", "sst_comp"]], get_Tw)["sst"]

### OHC

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


forced_wide, anom_wide = load_consolidated_wide()

In [ ]:
lon_avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean("longitude")

## compute ohc
Th["h_w_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 180)),
)["T"]

## compute ohc
Th["h_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 280)),
)["T"]


## compute ml temp
Th["T_3_ml"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.sel(z_t=slice(None, 50)).integrate("z_t"), (210, 270)),
)["T"]

### Surface heat flux

#### Load 

In [ ]:
flux_forced, flux_anom = src.utils.load_flux_data()

## switch sign convection to be positive into the ocean
for v in ["flns", "lhflx"]:
    flux_anom[v] = -1 * flux_anom[v]
    flux_forced[v] = -1 * flux_forced[v]

## add net heat flux data for convenience
flux_anom[["nhf", "nhf_comp"]] = anom[["nhf", "nhf_comp"]]

## load into memory
flux_anom.load();

### Compute regional averages

In [ ]:
def smooth_121(x):
    """apply 1-2-1 smoothing"""

    ## get 1-1-1 avg
    x_111_sum = x.rolling({"time": 3}, center=True, min_periods=1).sum()

    return 1 / 4 * (x_111_sum + x)


def get_Q(data, label, fn):
    """get heat flux over region using specified fn"""

    ### reconstruct flux over region
    hf = src.utils.reconstruct_wrapper(data, fn=fn)

    ## rolling mean (more comparable to dT/dt estimates?)
    # hf = smooth_121(hf)

    ## convert units to K/yr
    sec_per_mo = 8.64e4 * 30
    sec_per_yr = 12 * sec_per_mo
    rho = 1.02e3
    Cp = 4.2e3
    H = 50
    # H = 80

    ## apply scaling fac.
    Q = hf * sec_per_yr / (rho * Cp * H)

    return Q.rename({v: f"{v}_{label}" for v in list(hf)})

In [ ]:
## empty list to hold result
Q = []

for fn, label in tqdm.tqdm(
    [
        (src.utils.get_nino3, "3"),
        (src.utils.get_nino34, "34"),
        (get_Te, "e"),
        (get_nino_merged, "m"),
    ]
):
    Q.append(get_Q(flux_anom, label=label, fn=fn))

## list to xr
Q = xr.merge(Q)

## add to T, h data
Th = xr.merge([Th, Q])

### load 3-D data

In [ ]:
Th_3d = xr.open_mfdataset(
    sorted(list(pathlib.Path(DATA_FP / "cesm" / "Th_3D").glob("*nc"))),
    combine="nested",
    concat_dim="member",
)

## update coordinates to match
Th_3d = Th_3d.assign_coords({"member": np.arange(100)}).compute()
Th_3d = Th_3d.assign_coords({"time": Th.time})
Th_3d = Th_3d.rename({"h": "thermocline"})

In [ ]:
MLD = 50
avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean(["longitude"])
Th_3d["T_3"] = avg(Th_3d[f"T_{MLD}"], (210, 270))
Th_3d["T_34"] = avg(Th_3d[f"T_{MLD}"], (190, 240))
Th_3d["T_4"] = avg(Th_3d[f"T_{MLD}"], (160, 210))
Th_3d["T_e"] = avg(Th_3d[f"T_{MLD}"], (230, 280))
Th_3d["T_w"] = avg(Th_3d[f"T_{MLD}"], (180, 230))
Th_3d["h"] = avg(Th_3d["thermocline"], (120, 280))
Th_3d["h_w"] = avg(Th_3d["thermocline"], (120, 180))
Th_3d = Th_3d.drop_vars(["thermocline", "T_50", "T_80"])

### Standardize/window

In [ ]:
if USE_ML_AVG:
    for v in ["T_3", "T_34", "T_4", "T_e", "T_w"]:
        Th[v] = Th_3d[v]

## update thermocline variable
Th["h_mg"] = Th_3d["h"]
Th["h_w_mg"] = Th_3d["h_w"]

## non-dimensionalize
T_scale = Th["T_3"].std().values.item()
h_scale = Th["h_w"].std().values.item()

## non-dim T variables
for v in ["T_3", "T_34", "T_4", "T_e", "T_m"] + list(Q):
    Th[v] = Th[v] / T_scale


## non-dim h variables
for v in ["h", "h_w", "h_w_ohc", "h_ohc"]:
    Th[v] = Th[v] / h_scale

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)
# Th_rolling = src.utils.get_windowed(Th_3d, window_size=480, stride=120)

## drop last year because of NaNs
Th_rolling = Th_rolling.sel(year=slice(None, 2080))

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

## Remove linear dependence

Note: power point results use ```T_var=T_m```

In [ ]:
## remove SST dependence
for h_var in tqdm.tqdm(["h", "h_w", "h_ohc", "h_w_ohc", "h_mg", "h_w_mg"]):
    Th_rolling[f"{h_var}_hat"] = src.utils.remove_sst_dependence_v2(
        Th=Th_rolling,
        h_var=h_var,
        T_var=varnames[0],
        dims=["time", "member", "year"],
    )

### remove median

In [ ]:
## remove median (optional)
if REMOVE_MEDIAN:
    median = Th_rolling.groupby("time.month").median(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - median

else:
    mean = Th_rolling.groupby("time.month").mean(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - mean

## add constant
Th_rolling["ones"] = xr.ones_like(Th_rolling["T_3"])

## Compute time-varying RO parameters

### Fit model

Filenames:
- "T_3-h_w_ohc-zero_median_all-members.nc": ```maskb=["h"], maskNT=["T2", "T3", "TH"], maskNH=["T2"]```
- "T_3-h_w_ohc-zero_median.nc": same, but separate fits for each ensemble member
- "T_3-h_w_ohc-zero_median_simple.nc": ```maskNT=["T2", "T3"]```

In [ ]:
## parameters for fitting
# MODEL = src.XRO.XRO(
#     ncycle=12, ac_order=3, taus=np.zeros(4, dtype=int), is_forward=False
# )
MODEL = src.XRO.XRO(ncycle=12, ac_order=3, is_forward=False)

## nonlinear R
fit_kwargs = dict(
    ac_mask_idx=None,
    # maskNT=["T2", "T3", "H2"],
    maskNT=["T2", "T3"],
    # maskNT=["T2", "T3", "H2"],
    maskNH=["T2"],
)

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=False,
    fname=None,
    # fname=f"{varnames[0]}-{varnames[1]}-zero_median_simple.nc",
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

## compute error
fits["error"] = fits["Yfit"] - fits["Y"]
fits["error_amp"] = np.sqrt(fits["error"] ** 2)

## get corrected coords
time_coord = Th_rolling.stack(s=["member", "time"]).s
fits = fits.rename({"time": "s"}).assign_coords({"s": time_coord}).unstack("s")

#### Look at covariance between parameters

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(10, 7.5), layout="constrained")


## loop thru years
for yi, y in enumerate([1870, 2010, 2080]):

    ## get data for given year
    # Th_y = Th_rolling.sel(year=y).resample({"time":"QS-JAN"}).mean()
    Th_y = Th_rolling.sel(year=y)

    ## loop thru months
    for mi, m_idx in enumerate(np.arange(1, 13, 3)):

        ## subset for month
        is_m = Th_y.time.dt.month == m_idx
        Th_ym = Th_y.isel(time=is_m)

        ## plot data
        axs[yi, mi].scatter(Th_ym[varnames[0]], Th_ym[varnames[1]], s=2)

        ## formatting
        ax_kwargs = dict(ls="--", lw=1, c="gray")
        axs[yi, mi].axvline(0, **ax_kwargs)
        axs[yi, mi].axhline(0, **ax_kwargs)

src.utils.set_lims(axs)
for ax in axs[:-1].flatten():
    ax.set_xticks([])
for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
plt.show()

### validate

In [ ]:
# simulation specs
simulation_kwargs = dict(
    nyear=40,
    ncopy=500,
    seed=1000,
    X0_ds=Th_rolling[varnames].isel(year=0, member=0, time=0),
    # X0_ds = None,
    noise_type="white",
    use_noise_cov=True,
    is_xi_stdac=True,
    # noise_scale = .6,
    # noise_scale=1.25,  ## use for h_hat, is_forward=True
    # noise_scale=1.35, ## use for h_w
    noise_scale=1.45,  ## use for h_hat
)

In [ ]:
## do simulations
sims = get_sims_over_time(model=MODEL, params=fits, **simulation_kwargs)

## resample to DJF
get_djf = lambda x: x.resample({"time": "QS-DEC"}).mean().isel(time=slice(4, -4, 4))

## resample to seasonal
sims_djf = get_djf(sims)
Th_djf = get_djf(Th_rolling[varnames])

### Compute stats

In [ ]:
def get_std(x):
    return x.std(["time", "member"])


def get_quantiles(x):
    return x.quantile(dim=["time", "member"], q=[0.05, 0.1, 0.5, 0.9, 0.95])


def get_stats(x):
    """compute relevant stats for given dataset"""

    ## empty dataset to hold results
    stats = xr.Dataset()

    ## helper func to convert to xr.DataArray
    to_da = lambda x: x.to_dataarray(dim="v")

    ## compute
    stats["sigma"] = to_da(get_std(x))
    stats["q"] = to_da(get_quantiles(x))

    return stats

In [ ]:
stats_ro = get_stats(sims_djf)
stats_gt = get_stats(Th_djf)

### Plot results

#### Quantiles

In [ ]:
for varname in varnames:

    fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")
    for ax, stats_ in zip(axs, [stats_gt, stats_ro]):
        q = stats_["q"].sel(v=varname)

        ax.plot(stats_.year, q.sel(quantile=0.95), c="r", label="El Niño")
        ax.plot(stats_.year, -q.sel(quantile=0.05), c="b", label="La Niña")
        ax.plot(
            stats_.year,
            0.5 * (q.sel(quantile=0.95) - q.sel(quantile=0.05)),
            label="Average",
            c="k",
        )

        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        src.utils.add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010])
        # ax.set_ylim([0, None])

    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[1].legend()
    axs[0].set_title("CESM2")
    axs[1].set_title("RO")
    axs[0].set_ylabel(r"K")

    save(fig, fname=f"RO-valid_{varname}", do_save=False)
    plt.show()

In [ ]:
Th_ = Th_rolling.resample({"time": "QS-JAN"}).mean().isel(time=slice(4, -4, 4))

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
for ax, yr in zip(axs, [2010, 2080]):
    axs[0].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_w"].sel(year=yr),
        s=3,
    )
    axs[1].scatter(
        Th_["T_e"].sel(year=yr),
        Th_["T_e"].sel(year=yr) - Th_["T_w"].sel(year=yr),
        s=3,
    )

src.utils.set_lims(axs)

#### PDFs

In [ ]:
def get_pdf(x, year, varname=varnames[0], amp=4):
    """compute pdf for given data"""

    ## get edges
    edges = np.linspace(-amp, amp, 20)

    ## reshape data
    x_ = x[varname].sel(year=year).stack(s=["time", "member"]).values

    ## compute
    pdf, _ = src.utils.get_empirical_pdf(x_, edges=edges)

    return pdf, edges

In [ ]:
for PLOT_VAR, a in zip(varnames, [4, 2.5]):

    fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

    for ax, x, title in zip(axs, [Th_djf, sims_djf], ["CESM2", "RO"]):

        for y, kwargs in zip(
            [1870, 2010, 2080],
            [dict(color="gray", fill=True, alpha=0.2), dict(lw=2), dict(lw=2)],
        ):

            ## compute pdf for given year
            pdf_y, edges = get_pdf(x, year=y, varname=PLOT_VAR, amp=a)

            ## plot
            ax.stairs(pdf_y, edges, label=y, **kwargs)

        ## label/format
        ax.set_title(title)
        # ax.set_xlim([-3.3, 3.3])
        # ax.set_xticks([-3, 0, 3])
        ax_kwargs = dict(ls="--", c="k", lw=0.8)
        ax.axvline(0, **ax_kwargs)
        ax.axvline(-1.5, **ax_kwargs)

    ## formatting
    src.utils.set_lims(axs)
    axs[1].set_yticks([])
    axs[1].legend(prop=dict(size=10))
    plt.show()

#### Variance/skewness over time

In [ ]:
get_sigma = lambda x: x.std("time").quantile(dim="member", q=[0.1, 0.5, 0.9])
sigma_mod = get_sigma(Th_djf)
sigma_sim = get_sigma(sims_djf)

In [ ]:
## compute stats
get_stat = lambda x, fn: xr.apply_ufunc(
    fn, x, input_core_dims=[["time"]], kwargs={"axis": -1}
)
get_stat_wrapper = lambda x, fn: get_stat(x, fn).quantile(
    dim="member", q=[0.1, 0.5, 0.9]
)

# median_ot = get_stat(Th_djf, np.median)
skew_mod = get_stat_wrapper(Th_djf, scipy.stats.skew)
skew_sim = get_stat_wrapper(sims_djf, scipy.stats.skew)
# std_ot = get_stat(Th_djf, np.std)
# m3_ot = skew_ot * (std_ot**3)

In [ ]:
## specify colors
colors = sns.color_palette()

for PLOT_VAR in varnames:

    ## plot setup
    fig, axs = plt.subplots(1, 2, figsize=(6.5, 3), layout="constrained")

    ## plot data
    for ax, stats in zip(axs, [[sigma_mod, sigma_sim], [skew_mod, skew_sim]]):
        for stat, c, ls, label in zip(stats, colors, ["-", "--"], ["CESM2", "RO"]):
            for q, lw in zip([0.5, 0.1, 0.9], [2, 1, 1]):
                label_ = label if (q == 0.5) else None
                ax.plot(
                    stat.year,
                    stat[PLOT_VAR].sel(quantile=q),
                    c=c,
                    lw=lw,
                    ls=ls,
                    label=label_,
                )

    src.utils.add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010])

    axs[0].legend()
    axs[1].axhline(0, ls="--", c="k", lw=0.8)
    axs[0].set_ylabel("K")
    axs[1].set_ylabel("non-dim")
    axs[0].set_title(r"$\sigma(T)$ over time (DJF)")
    axs[1].set_title(r"skew$(T)$ over time (DJF)")

    save(fig, fname=f"RO-valid2_{PLOT_VAR}", do_save=False)
    plt.show()

In [ ]:
PLOT_VAR = varnames[0]

print(f"CESM2:")
fig, axs = plt.subplots(2, 1, figsize=(8, 3), layout="constrained")
axs[0].plot(np.arange(0, 40, 1 / 12), Th_rolling[PLOT_VAR].isel(member=9, year=10))
axs[1].plot(np.arange(0, 40, 1 / 12), Th_rolling[PLOT_VAR].isel(member=10, year=-1))
src.utils.set_lims(axs)
for ax in axs:
    ax.axhline(0, ls="--", c="k", lw=0.8)
plt.show()

print(f"\nSimulated:")
fig, axs = plt.subplots(2, 1, figsize=(8, 3), layout="constrained")
axs[0].plot(np.arange(0, 40, 1 / 12), sims[PLOT_VAR].isel(member=9, year=10))
axs[1].plot(np.arange(0, 40, 1 / 12), sims[PLOT_VAR].isel(member=10, year=-1))
src.utils.set_lims(axs)
for ax in axs:
    ax.axhline(0, ls="--", c="k", lw=0.8)
plt.show()

## look at params

In [ ]:
# delta =

xi = fits["xi_stdac"].isel(ranky=0)
R = fits["Lac"].isel(ranky=0, rankx=0) - fits["Lac"].isel(ranky=1, rankx=1)
F1 = fits["Lac"].isel(ranky=0, rankx=1)
F2 = -fits["Lac"].isel(ranky=1, rankx=0)
# beta= fits["NROT_Lac"].isel(nro_form=-1)
beta = fits["NROH_Lac"].isel(nro_form=0)
dxi = xi / xi.isel(year=0) - 1
dR = R / R.isel(year=0) - 1

fig, axs = plt.subplots(1, 4, figsize=(10, 2.5))
for ax, p in zip(axs, [R, xi, F1, beta]):

    for y in [1870, 1910, 1950, 1990]:

        ax.plot(params.cycle, p.sel(year=y))

axs[1].set_ylim([0, None])
axs[2].set_ylim([0, None])
src.utils.add_vticks(axs, xlines=[4, 6], xticks=[4, 6])

In [ ]:
BJ = fits["Lac"].isel(rankx=0, ranky=0) + fits["Lac"].isel(rankx=1, ranky=1)

fig, axs = plt.subplots(10, 1, figsize=(3, 11), layout="constrained")
for y in [1880, 1990]:  # , 2030, 2080]:
    axs[0].plot(fits.cycle, fits["Lac"].isel(rankx=0, ranky=0).sel(year=y), label=y)
    axs[1].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[2].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=2).sel(year=y), label=y)
    axs[3].plot(fits.cycle, fits["Lac"].isel(ranky=0, rankx=1).sel(year=y), label=y)
    axs[4].plot(fits.cycle, fits["NROT_Lac"].isel(nro_form=-1).sel(year=y), label=y)
    axs[5].plot(fits.cycle, fits["xi_stdac"].isel(ranky=0).sel(year=y), label=y)
    axs[6].plot(fits.cycle, -fits["Lac"].isel(ranky=1, rankx=0).sel(year=y), label=y)
    axs[7].plot(fits.cycle, -fits["NROH_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[8].plot(fits.cycle, -fits["Lac"].isel(rankx=1, ranky=1).sel(year=y), label=y)
    # axs[9].plot(fits.cycle, -fits["NLb_Lac"].isel(ranky=1).sel(year=y), label=y)
    axs[9].plot(fits.cycle, BJ.sel(year=y), label=y)


src.utils.add_vticks(axs, xlines=fits.cycle.values[[3, 5]], xticks=[])
kwargs = dict(x=1.0, y=0.5)
axs[0].text(s=r"$R$", transform=axs[0].transAxes, **kwargs)
axs[1].text(s=r"$\frac{dT}{dt}\propto T^2$", transform=axs[1].transAxes, **kwargs)
axs[2].text(s=r"$\frac{dT}{dt}\propto T^3$", transform=axs[2].transAxes, **kwargs)
axs[3].text(s=r"$F_1$", transform=axs[3].transAxes, **kwargs)
axs[4].text(s=r"$\frac{dT}{dt}\propto h^2$", transform=axs[4].transAxes, **kwargs)
axs[5].text(s=r"$\xi_T$", transform=axs[5].transAxes, **kwargs)
axs[6].text(s=r"$F_2$", transform=axs[6].transAxes, **kwargs)
axs[7].text(s=r"$\frac{dh}{dt}\propto -T^2$", transform=axs[7].transAxes, **kwargs)
axs[8].text(s=r"$\varepsilon$", transform=axs[8].transAxes, **kwargs)
# axs[9].text(s=r"$\frac{dh}{dt}\propto -h^2$", transform=axs[9].transAxes, **kwargs)
axs[9].text(s=r"$R-\varepsilon$", transform=axs[9].transAxes, **kwargs)
for ax in axs:
    ax.axhline(0, ls="--", c="gray", lw=0.8)
# ax.legend()

axs[-1].set_xticks(fits.cycle.values[[3, 5]], labels=[4, 6])
plt.show()

#### Hovmoller plot of $R$ and $\varepsilon$

In [ ]:
R_ = fits["Lac"].isel(rankx=0, ranky=0)
F1_ = fits["Lac"].isel(rankx=1, ranky=0)
R2 = fits["NROT_Lac"].isel(nro_form=0)
R3 = fits["NROT_Lac"].isel(nro_form=2)
eps_ = fits["Lac"].isel(rankx=1, ranky=1)
error_sigma = fits["xi_stdac"].isel(ranky=0)
# error_bar = fits["error"].groupby("time.month").mean(["time","member"]).isel(ranky=0)
# error_sigma =  fits["error"].groupby("time.month").std(["time","member"]).isel(ranky=0).rename({"month":"cycle"}).assign_coords({"cycle":R3.cycle})
# error_bar = (error_bar/error_sigma).rename({"month":"cycle"}).assign_coords({"cycle":R3.cycle})

In [ ]:
delta = lambda x: x - x.isel(year=0)
fig, axs = plt.subplots(1, 6, figsize=(12, 2), layout="constrained")

for ax, p, l in zip(
    axs,
    [R_, R_ + eps_, F1, R3, R2, error_sigma],
    [
        r"$R$",
        r"$R-\varepsilon$",
        r"$F_1$",
        r"$\frac{dT}{dt}\propto T^3$",
        r"$\frac{dT}{dt}\propto T^2$",
        "error",
    ],
):

    ax.contourf(
        p.cycle,
        p.year,
        delta(p),
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(0.75, 0.075),
        extend="both",
    )
    ax.axhline(1990, ls="--", c="gray", lw=1)
    ax.set_yticks([])
    ax.set_title(l)

src.utils.add_vticks(
    axs,
    xticks=[],
    xlines=[p.cycle[i] for i in [0, 3, 5]],
)


axs[0].set_yticks([1870, 1990, 2080])
plt.show()

In [ ]:
## check fits make sense
idx = dict(year=2, member=0, time=slice(None, 120))

## make plot
fig, ax = plt.subplots(figsize=(3, 2.5))
ax.plot(fits["Y"].isel(ranky=0, **idx))
ax.plot(get_ddt(Th_rolling[varnames[0]].isel(**idx)), ls="--")
plt.show()

In [ ]:
## resample to AMJ
fit_amj = fits.resample({"time": "QS-JAN"}).mean().isel(time=slice(1, -4, 4))
fit_a = fits.isel(time=slice(3, -12, 12))
fit_amj0 = fit_amj.sel(year=1880)
fit_amj1 = fit_amj.sel(year=1990)
fit_a0 = fit_a.sel(year=1880)
fit_a1 = fit_a.sel(year=1990)

In [ ]:
for fit_amj_, fit_a_ in zip([fit_amj0, fit_amj1], [fit_a0, fit_a1]):

    fig, axs = plt.subplots(2, 2, figsize=(5, 5), layout="constrained")

    ## loop thru (actual, error)
    for j, v in enumerate(["Y", "error"]):

        for xi, ax in enumerate(axs[j]):

            x0 = fit_a_["X"].isel(rankx=xi)
            # x1 = fit_a_[v].isel(ranky=0).transpose(*x0.dims)
            # x0 = fit_amj_["X"].isel(rankx=xi)
            x1 = fit_amj_[v].isel(ranky=0).transpose(*x0.dims)

            ax.scatter(x0, x1, s=3, alpha=0.5)
            corr = xr.corr(x0, x1).values.item()
            ax.set_title(f"Corr: {corr:.2f}")

            ## format
            ax_kwargs = dict(ls="--", c="k", lw=0.8)
            ax.axhline(0, **ax_kwargs)
            ax.axvline(0, **ax_kwargs)

        src.utils.set_lims(axs[j])
        axs[j, 1].set_yticks([])

    for ax in axs[0]:
        ax.set_xticks([])

    plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

for ax, year in zip(axs, [1880, 1990]):
    ax.scatter(
        fit_amj["Y"].isel(ranky=0).sel(year=year),
        fit_amj["Yfit"].isel(ranky=0).sel(year=year),
        s=3,
        alpha=0.5,
    )

    ## format
    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    zz = np.linspace(-10, 5)
    ax.plot(zz, zz, ls="-", lw=0.8, c="k")

axs[1].set_yticks([])
src.utils.set_lims(axs)

plt.show()

## Extract params

Get coefficients

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps = -fits["Lac"].isel(ranky=1, rankx=1)
coefs = xr.concat([R, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

Compute feature matrix for reconstructing nonlinear $R$

In [ ]:
## how should we scale the feature matrix?
## One of {"constant", "by_month", "sliding_by_month"}
SCALE_TYPE = "by_month"

## get scale
sigma_T = Th_rolling["T_3"].groupby("time.month").std(["time", "member"])

## average as needed
if SCALE_TYPE == "constant":
    sigma_T = sigma_T.mean(["year", "month"]) * xr.ones_like(sigma_T)

elif SCALE_TYPE == "by_month":
    sigma_T = sigma_T.mean("year") * xr.ones_like(sigma_T)

## update coordinate names to match
sigma_T = sigma_T.rename({"month": "cycle"}).assign_coords({"cycle": fits["cycle"]})

## scale feature mat by sigma_T
z = np.arange(-3, 3.1, 0.1)
z = xr.DataArray(z, coords=dict(sigma=z))
Z = sigma_T * z.expand_dims({"year": fits.year, "cycle": sigma_T.cycle})

## expand to get coefficients
Z = xr.concat([Z**i for i in range(3)], dim=coefs.form)

## check everything makes sense
print(np.allclose(Z.sel(form="T2").sel(sigma=1, method="nearest"), sigma_T))

Do the reconstruction

In [ ]:
## reconstruct nonlinear R
R_nl = (coefs * Z).sum("form")
R_nl_ann = R_nl.mean("cycle")
eps_ann = eps.mean("cycle")

## Compute time-varying $R_o$ and $\alpha$

### $\alpha$

#### Do fitting (or load pre-computed)

In [ ]:
def regress_cov_lagged(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    prep = lambda x: x.stack(s=["time", "member"]).transpose("year", ..., "s")

    ## get vars
    x_vars_plus = [f"{v}_plus" for v in x_vars]
    x_vars_minus = [f"{v}_minus" for v in x_vars]
    Y_ = prep(X[f"{y_var}_plus"])

    X_plus_ = prep(X[x_vars_plus].to_dataarray(dim="param"))
    X_minus_ = prep(X[x_vars_minus].to_dataarray(dim="param"))
    Y_plus_ = prep(X[f"{y_var}_plus"])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year": X.year, "param": x_vars},
        dims=["year", "param"],
    )

    ## covariance
    Cyx = np.einsum("yi,yji->yj", Y_plus_.values, X_minus_.values)
    Cxx = np.einsum("yji,yki->yjk", X_plus_.values, X_minus_.values)
    Cxx_inv = np.linalg.inv(Cxx)
    m.values = np.einsum("yj,yji->yi", Cyx, Cxx_inv)

    return m


def regress_cov_lagged_wrapper(X, x_vars, y_var, by_month=True):
    """helper func to get covariance"""

    ## Get xplus, xminus
    X_minus = X.isel(time=slice(None, -1))
    X_plus = X.isel(time=slice(1, None))
    X_plus = X_plus.assign_coords({"time": X_minus.time})
    X_minus = X_minus.rename({v: f"{v}_minus" for v in list(X_minus)})
    X_plus = X_plus.rename({v: f"{v}_plus" for v in list(X_plus)})
    X_ = xr.merge([X_plus, X_minus])

    ## now do regression
    kwargs = dict(x_vars=x_vars, y_var=y_var)
    if by_month:
        return X_.groupby("time.month").map(regress_cov_lagged, **kwargs)

    else:
        return regress_cov_lagged(X_, **kwargs)


def regress_pinv(X, x_vars, y_var):
    """do nonlinear regression"""

    ## prep data
    # prep = lambda x: x.transpose("member", ...)
    prep = lambda x: x.stack(s=["time", "member"]).transpose("year", ..., "s")

    X_ = prep(X[x_vars].to_dataarray(dim="v"))
    Y_ = prep(X[y_var])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"year": X.year, "v": x_vars},
        dims=["year", "v"],
    )

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("yi,yij->yj", Y_.values, X_pinv)

    return m.rename({"v": "param"})


def regress_pinv_wrapper(X, x_vars, y_var):
    return X.groupby("time.month").map(regress_pinv, x_vars=x_vars, y_var=y_var)


def regress_smooth(X, x_vars, y_var):
    """fit with harmonics"""

    ## get coefficients
    m = src.utils.regress_harm_wrapper(
        data=X,
        y_vars=[y_var],
        x_vars=x_vars,
        max_order=3,
        dims=["time", "member"],
    )

    ## convert from dataset to dataarray
    m = m.to_dataarray().squeeze(drop=True)

    return m.rename({"ell": "param"})


def get_alpha(
    X, x_vars, regress_fn, T_REGION="3", flux_types=["nhf", "fsns", "lhflx", "flns"]
):
    """compute alpha over time"""

    ## empty array to hold results
    alpha = []

    ## loop thru flux types
    for ftype in flux_types:

        kwargs = dict(x_vars=["T1", "T2", "T3"], y_var=f"{ftype}_{T_REGION}")
        alpha.append(regress_fn(X, **kwargs))

    ## conver to xr
    alpha = xr.concat(alpha, dim=pd.Index(flux_types, name="flux_type"))

    ## get points for evaluating alpha
    sigma = np.arange(-3, 3.1, 0.1)
    z = xr.DataArray(sigma, coords=dict(sigma=sigma))
    zz = xr.concat([z**i for i in range(1, 4)], dim=alpha.param)

    ## evaluate
    psi_eval = (zz * alpha).sum("param")

    ## merge
    alpha = xr.merge(
        [
            -alpha.rename("params"),
            psi_eval.rename("psi_eval"),
            -psi_eval.differentiate("sigma").rename("eval"),
        ]
    )

    ## reconstruct net heat flux from fsns and lhflx
    nhf_recon = (
        alpha.sel(flux_type="fsns")
        + alpha.sel(flux_type="lhflx")
        + alpha.sel(flux_type="flns")
    )
    nhf_recon = nhf_recon.expand_dims({"flux_type": ["nhf_recon"]})
    alpha = xr.concat([alpha, nhf_recon], dim="flux_type")

    return alpha

To-do: clean this up

In [ ]:
## get flux types
flux_types = ["nhf", "fsns", "lhflx", "flns"]

## merge T and Q data
T_data = xr.merge([(Th_rolling["T_3"] ** i).rename(f"T{i}") for i in range(4)])
flux_data = Th_rolling[[f"{ftype}_3" for ftype in flux_types]]
X = xr.merge([T_data, flux_data])

## regression with lagged covariance matrix
alpha_lagged = get_alpha(
    X=X,
    x_vars=["T1", "T2", "T3"],
    regress_fn=regress_cov_lagged_wrapper,
)


## regression with no lag in cov. matrices
alpha0 = get_alpha(
    X=X,
    x_vars=["T1", "T2", "T3"],
    regress_fn=regress_pinv_wrapper,
)

## regression using harmonics
# alpha_smooth = get_alpha(
#     X=X,
#     x_vars=["T1", "T2", "T3"],
#     regress_fn=regress_smooth,
# )

#### Plot results

In [ ]:
## specify which alpha to use
ALPHA = alpha0

## update month index
ALPHA = ALPHA.rename({"month": "cycle"}).assign_coords({"cycle": fits.cycle})

In [ ]:
## specify year indices
yis = [0, 12, -1]

fig, axs_ = plt.subplots(4, 3, figsize=(8, 10), layout="constrained")

for ji, yi in enumerate(yis):

    ## get axes for plotting
    axs = axs_[:, ji]
    axs[0].set_title(Th_rolling.year.values[yi])

    ## indx for scatter
    idx_sc = dict(year=yi, time=slice(mi, None, 12))

    ## get T variable
    T_data = Th_rolling[varnames[0]].isel(idx_sc)

    for ax, flux_type in zip(axs, ALPHA.flux_type.values[:-1]):

        ## plot data
        ax.scatter(
            T_data,
            Th_rolling[f"{flux_type}_{T_REGION}"].isel(idx_sc),
            s=5,
        )

        ## plot fit
        ax.plot(
            ALPHA.sigma,
            ALPHA["psi_eval"].sel(flux_type=flux_type).isel(year=yi, cycle=mi),
            c="magenta",
            ls="-",
        )

        ## plot fit for first period
        ax.plot(
            ALPHA.sigma,
            ALPHA["psi_eval"].sel(flux_type=flux_type).isel(year=0, cycle=mi),
            c="k",
            ls="--",
        )

    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.axhline(0, ls="--", c="gray", lw=1)
        ax.axvline(0, ls="--", c="gray", lw=1)

## label/format axes
src.utils.set_lims(axs_)
for ax in axs_[-1]:
    ax.set_xticks([-3, 0, 3])
    ax.set_xlabel(r"$T$ (K)")
for ax in axs_[:, 0]:
    ax.set_yticks([-5, 0, 5])
    ax.set_ylabel(r"$Q$ (K yr$^{-1}$)")
for ax, label in zip(
    axs_[:, -1],
    ["Net", "Shortwave", "Latent", "Longwave"],
):
    ax.text(s=label, x=1.1, y=0.5, transform=ax.transAxes)

save(fig, fname="Q-comparison", do_save=False)

plt.show()

Scratch (compute sum...)

In [ ]:
# ## plot sum of latent and shortwave
# nhf_recon = (
#     Th_rolling[f"fsns_{T_REGION}"]
#     + Th_rolling[f"lhflx_{T_REGION}"]
#     + Th_rolling[f"flns_{T_REGION}"]
# )
# axs[-1].scatter(T_data, nhf_recon.isel(idx_sc), s=5)

# # ## plot reconstruction
# # alpha_recon = alpha.sel(flux_type="fsns") + alpha.sel(flux_type="lhflx")
# axs[-1].plot(
#     ALPHA.sigma,
#     ALPHA["psi_eval"].isel(year=yi, cycle=mi).sel(flux_type="nhf_recon"),
#     c="k",
# )
# axs[-1].set_title("fsns + lhflx")

### Compute $R_o$

In [ ]:
def get_fits_alpha(fits, alpha):
    """put alpha in fits format"""

    ## empty fits dataset
    fits_alpha = xr.zeros_like(fits)
    fits_alpha["var_names"] = fits["var_names"]

    ## helper func to get params
    get_p = lambda x: x["params"].transpose("year", "cycle").values

    ## update linear
    alpha_Lac = fits_alpha["Lac"].transpose("year", "cycle", "ranky", "rankx")
    alpha_Lac.values[..., 0, 0] = get_p(alpha.sel(param="T1"))

    ## update nonlinear
    alpha_non = fits_alpha["NROT_Lac"].transpose("year", "cycle", "nro_form")
    alpha_non.values[..., 0] = get_p(alpha.sel(param="T2"))
    alpha_non.values[..., 2] = get_p(alpha.sel(param="T3"))

    ## update fits
    fits_alpha["Lac"] = alpha_Lac.transpose(*fits_alpha["Lac"].dims)
    fits_alpha["NROT_Lac"] = alpha_non.transpose(*fits_alpha["NROT_Lac"].dims)

    return fits_alpha


def op_fits(fits1, fits0, op):
    """Handle operation on two fits objects, handling non-numeric data"""

    ## get list of valid variables
    valid_vars = [x for x in list(fits0) if (x != "var_names")]

    ## subtract
    res = op(fits1[valid_vars], fits0[valid_vars])

    ## update other variables
    res["var_names"] = fits0["var_names"]

    return res


def sum_fits(fits1, fits0):
    """Get difference between fits (fits1-fits0).
    Handles non-numeric data"""

    return op_fits(fits1, fits0, op=lambda x1, x0: x1 + x0)


def diff_fits(fits1, fits0):
    """Get difference between fits (fits1-fits0).
    Handles non-numeric data"""

    return op_fits(fits1, fits0, op=lambda x1, x0: x1 - x0)

In [ ]:
## evaluated
Ro = R_nl + ALPHA["eval"].sel(flux_type="nhf")

## get alpha and Ro in fits format
fits_alpha = get_fits_alpha(fits=fits, alpha=ALPHA.sel(flux_type="nhf"))
fits_Ro = sum_fits(fits1=fits, fits0=fits_alpha)

## same, but for shortwave and lhflx
fits_alpha_sw = get_fits_alpha(fits=fits, alpha=ALPHA.sel(flux_type="fsns"))
fits_minus_sw = sum_fits(fits1=fits, fits0=fits_alpha_sw)

fits_alpha_lhf = get_fits_alpha(fits=fits, alpha=ALPHA.sel(flux_type="lhflx"))
fits_minus_lhf = sum_fits(fits1=fits, fits0=fits_alpha_lhf)

fits_alpha_lw = get_fits_alpha(fits=fits, alpha=ALPHA.sel(flux_type="flns"))
fits_minus_lw = sum_fits(fits1=fits, fits0=fits_alpha_lw)

Check it works

In [ ]:
print(
    np.allclose(
        ALPHA["params"].sel(param="T1", flux_type="nhf").transpose("year", ...),
        (fits_Ro["Lac"] - fits["Lac"]).isel(rankx=0, ranky=0),
    )
)
print(
    np.allclose(
        ALPHA["params"].sel(param="T2", flux_type="nhf").transpose("year", ...),
        (fits_Ro["NROT_Lac"] - fits["NROT_Lac"]).isel(nro_form=0),
    )
)
print(
    np.allclose(
        ALPHA["params"].sel(param="T3", flux_type="nhf").transpose("year", ...),
        (fits_Ro["NROT_Lac"] - fits["NROT_Lac"]).isel(nro_form=2),
    )
)

print(
    np.allclose(
        fits_alpha[["Lac", "NROT_Lac"]].to_dataarray(),
        (fits_Ro[["Lac", "NROT_Lac"]] - fits[["Lac", "NROT_Lac"]]).to_dataarray(),
    )
)

### Hovmoller plot of $R_o$ and $\varepsilon$

In [ ]:
Ro_ = fits_Ro["Lac"].isel(rankx=0, ranky=0)
Ra_ = fits_alpha["Lac"].isel(rankx=0, ranky=0)
# F1_ = fits_Ro["Lac"].isel(rankx=1,ranky=0)
Ro2 = fits_Ro["NROT_Lac"].isel(nro_form=0)
Ro3 = fits_Ro["NROT_Lac"].isel(nro_form=2)

In [ ]:
delta = lambda x: x - x.isel(year=0)
fig, axs = plt.subplots(1, 5, figsize=(10, 2), layout="constrained")

for ax, p, l in zip(
    axs,
    [R_, Ro_, -Ra_, Ro3, Ro2],
    [
        r"$R$",
        r"$R_o$",
        r"$-\alpha$",
        r"$\frac{dT}{dt}\propto T^3$",
        r"$\frac{dT}{dt}\propto T^2$",
    ],
):

    ax.contourf(
        p.cycle,
        p.year,
        delta(p),
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(0.75, 0.075),
        extend="both",
    )
    ax.axhline(1990, ls="--", c="gray", lw=1)
    ax.set_yticks([])
    ax.set_title(l)

src.utils.add_vticks(
    axs,
    xticks=[],
    xlines=[p.cycle[i] for i in [0, 3, 5]],
)


axs[0].set_yticks([1870, 1990, 2080])
plt.show()

Cleaned up plot

In [ ]:
delta = lambda x: x - x.isel(year=0)
fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

Ra_3 = fits_alpha["NROT_Lac"].isel(nro_form=0)

for ax, p, l, amp in zip(
    axs,
    [R_, Ro_, -Ra_, error_sigma],
    [
        r"$R$",
        r"$R_o$",
        r"$-\alpha$",
        r"Noise amplitude",
    ],
    [1, 1, 1, 0.5],
):

    cp = ax.contourf(
        p.cycle,
        p.year,
        delta(p),
        cmap="cmo.balance",
        levels=src.utils.make_cb_range(amp, amp / 10),
        extend="both",
    )
    ax.axhline(1990, ls="--", c="gray", lw=1)
    ax.set_yticks([])
    ax.set_title(l)
    ax.set_xticks([p.cycle[i] for i in [0, 5]], labels=["Jan", "Jun"])
    ax.axvline(p.cycle[5], ls="--", c="gray", lw=0.8)

## label
fig.colorbar(cp, ax=axs[-2], ticks=[-1, 0, 1], label=r"yr$^{-1}$")
fig.colorbar(cp, ax=axs[-1], ticks=[-0.5, 0, 0.5], label=r"K$^{-1}$ yr$^{-1}$")
axs[0].set_yticks([1870, 1990, 2080])

## Save
save(fig, fname="param-hov", do_save=False)

plt.show()

In [ ]:
BJ = fits_Ro["Lac"].isel(rankx=0, ranky=0) + fits_Ro["Lac"].isel(rankx=1, ranky=1)

fig, axs = plt.subplots(10, 1, figsize=(3, 11), layout="constrained")
for y in [1880, 1990, 2030]:  # , 2030, 2080]:
    # axs[0].plot(fits.cycle, fits_Ro.sel(year=y), label=y)
    axs[0].plot(fits.cycle, Ro_.sel(year=y), label=y)
    axs[1].plot(fits.cycle, fits_Ro["NROT_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[2].plot(fits.cycle, fits_Ro["NROT_Lac"].isel(nro_form=2).sel(year=y), label=y)
    axs[3].plot(fits.cycle, fits_Ro["Lac"].isel(ranky=0, rankx=1).sel(year=y), label=y)
    axs[4].plot(fits.cycle, fits_Ro["NROT_Lac"].isel(nro_form=-1).sel(year=y), label=y)
    axs[5].plot(fits.cycle, fits_Ro["xi_stdac"].isel(ranky=0).sel(year=y), label=y)
    axs[6].plot(fits.cycle, -fits_Ro["Lac"].isel(ranky=1, rankx=0).sel(year=y), label=y)
    axs[7].plot(fits.cycle, -fits_Ro["NROH_Lac"].isel(nro_form=0).sel(year=y), label=y)
    axs[8].plot(fits.cycle, -fits_Ro["Lac"].isel(rankx=1, ranky=1).sel(year=y), label=y)
    # axs[9].plot(fits.cycle, -fits["NLb_Lac"].isel(ranky=1).sel(year=y), label=y)
    axs[9].plot(fits.cycle, BJ.sel(year=y), label=y)


src.utils.add_vticks(axs, xlines=fits.cycle.values[[3, 5]], xticks=[])
kwargs = dict(x=1.0, y=0.5)
axs[0].text(s=r"$R$", transform=axs[0].transAxes, **kwargs)
axs[1].text(s=r"$\frac{dT}{dt}\propto T^2$", transform=axs[1].transAxes, **kwargs)
axs[2].text(s=r"$\frac{dT}{dt}\propto T^3$", transform=axs[2].transAxes, **kwargs)
axs[3].text(s=r"$F_1$", transform=axs[3].transAxes, **kwargs)
axs[4].text(s=r"$\frac{dT}{dt}\propto h^2$", transform=axs[4].transAxes, **kwargs)
axs[5].text(s=r"$\xi_T$", transform=axs[5].transAxes, **kwargs)
axs[6].text(s=r"$F_2$", transform=axs[6].transAxes, **kwargs)
axs[7].text(s=r"$\frac{dh}{dt}\propto -T^2$", transform=axs[7].transAxes, **kwargs)
axs[8].text(s=r"$\varepsilon$", transform=axs[8].transAxes, **kwargs)
# axs[9].text(s=r"$\frac{dh}{dt}\propto -h^2$", transform=axs[9].transAxes, **kwargs)
axs[9].text(s=r"$R-\varepsilon$", transform=axs[9].transAxes, **kwargs)
for ax in axs:
    ax.axhline(0, ls="--", c="gray", lw=0.8)
# ax.legend()

axs[-1].set_xticks(fits.cycle.values[[3, 5]], labels=[4, 6])
plt.show()

## Plot params over time

In [ ]:
def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return

Plot snapshots of growth rate over time. Also, plot seasonal cycle.

In [ ]:
## specify where to evaluate R value
T0 = 1.5

if T0 is None:
    """for this case: average over all pos/neg values"""
    R_pos = R_nl.where(R_nl["sigma"] > 0).mean("sigma")
    R_neg = R_nl.where(R_nl["sigma"] < 0).mean("sigma")
    R_all = R_nl.mean("sigma")

else:
    """for this case: find R at given sigma value"""
    R_pos = R_nl.sel(sigma=T0, method="nearest")
    R_neg = R_nl.sel(sigma=-T0, method="nearest")
    R_all = R_nl.sel(sigma=0, method="nearest")

In [ ]:
YEARS = np.array([1880, 2010, 2050, 2080])
colors = cmocean.cm.balance(np.linspace(0.2, 0.8, len(YEARS)))

fig, axs = plt.subplots(1, 3, figsize=(9, 2.5), layout="constrained")

for c, y in zip(colors, YEARS):

    ## plot R nonlinearity
    axs[0].plot(R_nl["sigma"], R_nl_ann.sel(year=y), c=c, label=y)
    # axs[0].plot(R_nl["sigma"], (xr.ones_like(R_nl_ann)-eps.mean("cycle")).sel(year=y), c=c, label=y)
    # axs[0].plot(R_nl["sigma"], (R_nl_ann - eps.mean("cycle")).sel(year=y), c=c, label=y)

    ## plot R seasonal cycle
    axs[1].plot(
        # params.cycle,
        # 0.5 * (R_pos - eps).sel(year=y).squeeze(drop=True),
        # # -0.5 * (eps).sel(year=y).squeeze(drop=True),
        # c=c,
        # label=y,
        params.cycle,
        R_pos.sel(year=y).squeeze(drop=True),
        c=c,
        label=y,
    )
    axs[2].plot(
        # params.cycle,
        # 0.5 * (R_neg - eps).sel(year=y).squeeze(drop=True),
        # c=c,
        # label=y,
        params.cycle,
        R_neg.sel(year=y).squeeze(drop=True),
        c=c,
        label=y,
    )


ax_kwargs = dict(ls="--", c="k", lw=0.8)
axs[0].axvline(0, **ax_kwargs)

axs[0].set_xlabel(r"$T$")
axs[0].set_ylabel(r"Growth rate (yr$^{-1}$)")
axs[0].set_title(r"$R(T)$, annual mean")
axs[1].set_title(f"R($T=${T0:.1f})")
axs[2].set_title(f"R($T=-${T0:.1f})")
axs[2].legend(loc=(1.1, 0.15))

for ax in axs:
    ax.axhline(0, **ax_kwargs)

add_vticks(axs[1:], xticks=[1, 7, 12], xlines=[7])
for ax in axs[1:]:
    ax.set_xlabel("Month")
    ax.set_xticks([1, 7, 12], labels=["Jan", "Jul", "Dec"])

src.utils.set_lims(axs[1:])
if T0 is not None:
    add_vticks(axs[:1], xticks=[-T0, 0, T0], xlines=[-T0, 0, T0])

save(fig, fname="gr-cycle", do_save=False)

plt.show()

In [ ]:
## select cycle
# SEL_CYCLE = lambda x: x.isel(cycle=slice(9, 12)).mean("cycle")
# SEL_CYCLE = lambda x: x.isel(cycle=[7,8,9]).mean("cycle")
# SEL_CYCLE = lambda x: x.isel(cycle=slice(6,None)).mean("cycle")
SEL_CYCLE = lambda x: x.mean("cycle")

fig, axs = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")

## plot R and R-epsilon
for R_, c, label in zip(
    [R_pos, R_neg, R_all], ["r", "b", "k"], [f"$T=${T0}", f"$T=-${T0}", "All"]
):

    ## get shared args
    plot_kwargs = dict(c=c, label=label)

    axs[0].plot(R_.year, SEL_CYCLE(R_), **plot_kwargs)
    axs[1].plot(R_.year, 0.5 * SEL_CYCLE(R_ - eps), **plot_kwargs)

## plot noise over time
delta = lambda x: x / x.isel(year=1) - 1
axs[2].plot(params.year, delta(params["xi_T"].mean("cycle")), label=r"$T$")
axs[2].plot(params.year, delta(params["xi_h"].mean("cycle")), label=r"$h$")
# axs[2].plot(params.year, params["xi_T"].mean("cycle"), label=r"$T$")
# axs[2].plot(params.year, params["xi_h"].mean("cycle"), label=r"$h$")

## label
labels = [
    r"$R$",
    r"$\frac{1}{2}\left(R-\varepsilon\right)$",
    r"$\Delta$ (stochastic forcing amp.)",
]
for ax, title in zip(axs, labels):
    ax.set_title(title)
axs[0].legend()
axs[2].legend()
# axs[2].set_yticks([0, 0.1, 0.2, 0.3])

## formatting
axs[0].set_ylabel(r"Growth rate (yr$^{-1}$)")
axs[2].set_ylabel(r"Fraction")
ax_kwargs = dict(ls="--", c="k", lw=0.8)
add_vticks(axs, xticks=YEARS[:-1], xlines=YEARS[:-1])
# for ax in axs[-1:]:
#     ax.axhline(0, **ax_kwargs)

axs[0].axhline(0, **ax_kwargs)
axs[0].set_yticks([-1, 0])
axs[1].set_yticks([-1, -0.5])
axs[2].set_yticks([0, 0.1])

save(fig, fname="gr-over-time_ann", do_save=False)

plt.show()

#### Comparison plot between $R,R_o,\alpha$

In [ ]:
def plot_p(ax, p, months):
    """plot change over time for parameter"""

    ## get month indexes
    month_idxs = np.array(months) - 1

    ## helper funcs
    sel_month = lambda x: x.isel(cycle=month_idxs).mean("cycle")
    center = lambda x: x - x.isel(year=0)
    get = lambda x: center(sel_month(x))
    # get = lambda x : sel_month(x)

    ## get colors for plotting
    colors = cmocean.cm.balance(np.linspace(0.2, 0.8, 4))
    colors = [colors[i] for i in range(colors.shape[0])]
    colors = colors[:2] + ["k"] + colors[2:]

    ## plot different T values
    for s_, color in zip([-1, -0.5, 0, 0.5, 1], colors):

        ## plot data
        ax.plot(
            p.year, get(p.sel(sigma=s_, method="nearest")), c=color, label=f"$T=${s_}"
        )

    return


def plot_p_comp(axs, p0, p1, p2, p3, months):
    """plot change over time for parameters"""

    ## plot total R, damping, and ocean contribution
    for ax, p in zip(axs, [p0, p1, p2, p3]):

        plot_p(ax, p, months=months)
        ax.set_ylim([-1.4, 1.4])

    axs[0].set_yticks([-1, 0, 1])

    src.utils.set_lims(axs)
    axs[0].set_ylabel(r"yr$^{-1}$")
    for ax in axs:
        ax.axhline(0, ls="--", c="k", lw=0.8)
    for ax in axs[1:]:
        ax.set_yticks([])

    src.utils.add_vticks(axs, xlines=[2010, 2060], xticks=[1870, 2010, 2060])

    return

In [ ]:
fig, axs = plt.subplots(5, 4, figsize=(10, 9), layout="constrained")

for j, (months, label, amp) in enumerate(
    zip(
        [[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12], np.arange(1, 13)],
        ["JFM", "AMJ", "JAS", "OND", "All"],
        [2, 2, 1.5, 1, 1],
        # [[12,1,2], [3,4,5], [6,7,8], [9,10,11], np.arange(1, 13)],
        # ["DJF", "MAM", "JJA", "SON", "All"],
        # [2, 2, 1.5, 1, 1],
    )
):

    plot_p_comp(
        axs[j],
        p0=0.5 * (R_nl - eps),
        p1=R_nl,
        p2=-ALPHA["eval"].sel(flux_type="nhf"),
        p3=Ro,
        months=months,
    )
    axs[j, -1].text(s=label, x=1.1, y=0.5, transform=axs[j, -1].transAxes)

for axs_ in axs[:-1]:
    for ax in axs_:
        ax.set_xticks([])

axs[0, 0].set_title(r"$\frac{1}{2}\left(R-\varepsilon\right)$")
axs[0, 1].set_title(r"$R$")
axs[0, 2].set_title(r"$-\alpha$")
axs[0, 3].set_title(r"$R_o=R+\alpha$")
axs[0, 0].legend(prop=dict(size=8), loc="lower left")


# src.utils.set_lims(axs.flatten())

save(fig, "gr-over-time", do_save=False)

plt.show()

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(10, 6), layout="constrained")

for j, (months, label) in enumerate(
    zip(
        [[1, 3, 4, 5, 6], [7, 8, 9, 10, 11, 12], np.arange(1, 13)],
        ["Jan-Jun", "Jul-Dec", "All"],
    )
):

    plot_p_comp(
        axs[j],
        p0=0.5 * (R_nl - eps),
        p1=R_nl,
        p2=-ALPHA["eval"].sel(flux_type="nhf"),
        p3=Ro,
        months=months,
    )
    axs[j, -1].text(s=label, x=1.1, y=0.5, transform=axs[j, -1].transAxes)

for axs_ in axs[:-1]:
    for ax in axs_:
        ax.set_xticks([])

axs[0, 0].set_title(r"$\frac{1}{2}\left(R-\varepsilon\right)$")
axs[0, 1].set_title(r"$R$")
axs[0, 2].set_title(r"$-\alpha$")
axs[0, 3].set_title(r"$R_o=R+\alpha$")


# src.utils.set_lims(axs.flatten())

plt.show()

#### Plot in a slightly different way

In [ ]:
# fits_Ro = fits[["Lac","NROT_Lac"]] - fits_fixed_Ro[["Lac","NROT_Lac"]]
fits_Ro_ = fits[["Lac", "NROT_Lac"]] + fits_alpha[["Lac", "NROT_Lac"]]

In [ ]:
yr = fits.year
sel = lambda x: x[["Lac", "NROT_Lac"]].mean("cycle")
delta = lambda x: x - x.isel(year=0)
sel2 = lambda x: delta(sel(x))

## funcs to select params
sel_sym = lambda x: x["Lac"].isel(rankx=0, ranky=0)
sel_asym = lambda x: x["NROT_Lac"].isel(nro_form=0)
sel_cub = lambda x: x["NROT_Lac"].isel(nro_form=2)


fig, axs = plt.subplots(1, 3, figsize=(9, 2.5), layout="constrained")

for ax, sel_fn in zip(axs, [sel_sym, sel_asym, sel_cub]):

    ## get func to select all data
    sel_ = lambda x: sel_fn(sel2(x))

    ## plot total
    ax.plot(yr, sel_(fits), c="k", label="$R$")

    ##
    for fits_, l in zip(
        [fits_Ro_, -fits_alpha[["Lac", "NROT_Lac"]]], [r"$R_o$", r"$-\alpha$"]
    ):
        ax.plot(yr, sel_(fits_), label=l)

for ax in axs:
    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **ax_kwargs)

src.utils.add_vticks(axs, xticks=[1870, 2010], xlines=[2010])
# axs[1].set_yticks([])

src.utils.set_lims(axs)
axs[0].set_ylabel(r"yr$^{-1}$")
axs[1].set_ylabel(r"K$^{-1}$ yr$^{-1}$")
axs[2].set_ylabel(r"K$^{-2}$ yr$^{-1}$")
axs[0].legend()
axs[0].set_title("Linear")
axs[1].set_title("Quadratic")
axs[2].set_title("Cubic")

plt.show()

## Ablation tests

### Funcs to tweak params

In [ ]:
def fix_R0(params, **kwargs):
    """Fix R parameter"""

    ## fix T (linear R)
    pparams = src.utils.get_perturbed_multi(params=params, idxs=[(0, 0)], **kwargs)

    return pparams


def fix_R2(params, **kwargs):
    """Fix quadratic R parameter"""

    ## fix T (quadratic R)
    pparams = get_perturbed_NRO(nro_form_idx=0, params=params, nro_type="NROT_Lac")

    return pparams


def fix_R3(params, **kwargs):
    """Fix quadratic R parameter"""

    ## fix T (quadratic R)
    pparams = get_perturbed_NRO(nro_form_idx=2, params=params, nro_type="NROT_Lac")

    return pparams


def fix_R(params, **kwargs):
    """Fix R parameter"""

    ## fix T (linear R)
    pparams0 = fix_R0(params=params, **kwargs)

    ## fix T2
    pparams1 = fix_R2(params=pparams0)

    ## fix T3
    pparams = fix_R3(params=pparams1)

    return pparams


def fix_F1(params, **kwargs):
    """Fix F1 parameter"""

    ## fix linear param (depependence on H)
    pparams0 = src.utils.get_perturbed_multi(params=params, idxs=[(0, 1)], **kwargs)

    ## fix dependence on H2
    pparams1 = get_perturbed_NRO(params=pparams0, nro_form_idx=-1, nro_type="NROT_Lac")

    ## fix dependence on T, H
    pparams = get_perturbed_NRO(params=pparams1, nro_form_idx=1, nro_type="NROT_Lac")

    return pparams


def fix_F2(params, **kwargs):
    """Fix F1 parameter"""

    ## fix linear param (depependence on H)
    pparams0 = src.utils.get_perturbed_multi(params=params, idxs=[(1, 0)], **kwargs)

    ## fix dependence on T2
    pparams = get_perturbed_NRO(params=pparams0, nro_form_idx=0, nro_type="NROH_Lac")

    return pparams


def fix_BJ(params, **kwargs):
    """fix R and epsilon"""

    ## get fixed R
    pparams_fixed_R = fix_R(params, **kwargs)

    ## fix epsilon
    pparams = src.utils.get_perturbed_multi(
        params=pparams_fixed_R, idxs=[(1, 1)], **kwargs
    )

    return pparams


def fix_wyrtki(params, **kwargs):
    """fix F1 and F2"""

    ## fix F1
    pparams_fixed_F1 = fix_F1(params, **kwargs)

    ## fix F2
    pparams = fix_F2(params=params_fixed_F1, **kwargs)

    return pparams


def fix_ddt_T(params, **kwargs):
    """fix R and F1"""

    ## get fixed R
    pparams_fixed_R = fix_R(params, **kwargs)

    ## fix F1
    pparams = fix_F1(params=pparams_fixed_R, **kwargs)

    return pparams


def fix_ddt_h(params, **kwargs):
    """fix epsilon and F2"""

    ## get fixed R
    pparams_fixed_F2 = fix_F2(params, **kwargs)

    ## fix F1
    pparams = fix_eps(params=pparams_fixed_F2, **kwargs)

    return pparams


def fix_eps(params, **kwargs):
    """fix epsilon"""
    pparams = src.utils.get_perturbed_multi(params=params, idxs=[(1, 1)], **kwargs)

    return pparams


def fix_cycle(params, pparams, cycle_idxs):
    """only keep perturbation for part of seasonal cycle"""

    ## these are param names to modify
    pnames = [
        "Lac",
        "NLb_Lac",
        "NLc_Lac",
        "NROT_Lac",
        "NROH_Lac",
        "xi_stdac",
        "xi_std",
        "xi_cov",
        "xi_covac",
    ]

    ## get difference in params
    dparams = fits[pnames] - pparams[pnames]

    ## get months to return to baseline values (difference b/n sets)
    idxs_to_zero = set(np.arange(12)) - set(cycle_idxs)

    ## convert from set to array
    idxs_to_zero = np.array(list(idxs_to_zero))

    ## now remove param perturbations from these indices
    dparams_updated = xr.merge(
        [
            dparams.isel(cycle=cycle_idxs).drop_vars("xi_cov"),
            xr.zeros_like(dparams.isel(cycle=idxs_to_zero)),
        ]
    )

    ## get updated param perturbations
    ## apply updated perturbation to original parameter set
    pparams_updated = copy.deepcopy(pparams)
    pparams_updated[pnames] = fits[pnames] - dparams_updated[pnames]

    return pparams_updated

Test they work

In [ ]:
fits_R = fix_R(fits)
fits_R0 = fix_cycle(params=fits, pparams=fits_R, cycle_idxs=np.arange(6))

sel = lambda x: x["Lac"].isel(rankx=0, ranky=0).mean("cycle")

fig, axs = plt.subplots(1, 2, figsize=(5, 2))

for ax, cyc_idx in zip(axs, [np.arange(0, 6), np.arange(6, 12)]):

    ax.plot(fits.year, sel(fits.isel(cycle=cyc_idx)), c="k")
    ax.plot(fits.year, sel(fits_R.isel(cycle=cyc_idx)))
    ax.plot(fits.year, sel(fits_R0.isel(cycle=cyc_idx)), ls="--")

src.utils.set_lims(axs)
plt.show()

### Specify ablation setup

In [ ]:
## should we fix the given parameter? or all others?
FIX_OTHERS = False
FIX_NOISE = False

## shared args
pparam_kwargs = dict(params=fits, fix_others=FIX_OTHERS, fix_noise=FIX_NOISE)

## get fixed params
fits_fixed_cubic = get_perturbed_NRO(nro_form_idx=2, params=fits, nro_type="NROT_Lac")
fits_fixed_quad = get_perturbed_NRO(nro_form_idx=0, params=fits, nro_type="NROT_Lac")
fits_fixed_R0 = fix_R0(**pparam_kwargs)
fits_fixed_R = fix_R(**pparam_kwargs)
fits_fixed_ddt_h = fix_ddt_h(**pparam_kwargs)
fits_fixed_ddt_T = fix_ddt_T(**pparam_kwargs)
fits_fixed_noise = src.utils.get_perturbed_noise(fits, fix_others=FIX_OTHERS)

## ocean/atm experiments
fits_fixed_Ro_F1 = diff_fits(
    fix_F1(fix_R(params=fits_Ro)),
    fits_alpha,
)

fits_fixed_Ro = diff_fits(
    fix_R(params=fits_Ro),
    fits_alpha,
)

fits_fixed_Ro_lin = diff_fits(
    fix_R0(params=fits_Ro),
    fits_alpha,
)

fits_fixed_Ro_cub = diff_fits(
    fix_R3(params=fits_Ro),
    fits_alpha,
)

fits_fixed_Ro_F1_lin = diff_fits(
    fix_F1(fix_R0(params=fits_Ro)),
    fits_alpha,
)

fits_fixed_alpha = diff_fits(
    fits_Ro,
    fix_R(fits_alpha),
)

fits_fixed_sw = diff_fits(
    fits_minus_sw,
    fix_R(fits_alpha_sw),
)

fits_fixed_lhf = diff_fits(
    fits_minus_lhf,
    fix_R(fits_alpha_lhf),
)

fits_fixed_lw = diff_fits(
    fits_minus_lw,
    fix_R(fits_alpha_lw),
)

## fix ocean terms
fits_fixed_cubic = fix_R3(**pparam_kwargs)
fits_fixed_quad = fix_R2(**pparam_kwargs)
fits_fixed_R0 = fix_R0(**pparam_kwargs)
fits_fixed_F1 = fix_F1(**pparam_kwargs)

## shared args (fix cycle)
# cyc_kwargs = dict(params=fits, pparams=fits_fixed_Ro)
# cyc_kwargs = dict(params=fits, pparams=fits_fixed_R0)
# cyc_kwargs = dict(params=fits, pparams=fits_fixed_alpha)
cyc_kwargs = dict(params=fits, pparams=fits_fixed_noise)

## specify params for experiments
param_set_dict = {
    "Control": fits,
    "Fix noise": fits_fixed_noise,
    r"Fix $R$, $F_1$": fits_fixed_ddt_T,
    r"Fix $R_o$, $F_1$": fits_fixed_Ro_F1,
    # r"Fix $R_o$, $F_1$": fits_fixed_Ro,
    r"Fix $R_o$": fits_fixed_Ro,
    r"Fix $R$": fits_fixed_R,
    r"Fix $F1$": fits_fixed_F1,
    r"Fix $F_2$, $\varepsilon$": fits_fixed_ddt_h,
    # r"Fix lin.": fits_fixed_R0,
    # r"Fix quad": fits_fixed_quad,
    # r"Fix cubic": fits_fixed_cubic,
    r"Fix $\alpha$": fits_fixed_alpha,
    r"Fix shortwave": fits_fixed_sw,
    r"Fix latent": fits_fixed_lhf,
    r"Fix longwave": fits_fixed_lw,
    # r"Fix $F_1$": fix_F1(**pparam_kwargs),
    # r"Fix $R$, $F_1$ (AMJ)": fix_cycle(cycle_idxs=np.arange(4,7), params=fits, pparams=fits_fixed_Ro_F1),
    # r"Fix $R_o$ (AMJ)": fix_cycle(cycle_idxs=np.arange(4,7), params=fits, pparams=fits_fixed_Ro),
    # r"Fix $R_0$ (lin, AMJ)": fix_cycle(cycle_idxs=np.arange(4,7), params=fits, pparams=fits_fixed_Ro_lin),
    # r"Fix $R_0$ (cub, AMJ)": fix_cycle(cycle_idxs=np.arange(4,7), params=fits, pparams=fits_fixed_Ro_cub),
    # r"Fix $F_1$ (AMJ)": fix_cycle(cycle_idxs=np.arange(4,7), params=fits, pparams=fits_fixed_F1),
    # r"Fix $R$, $F_1$ (JFM)": fix_cycle(cycle_idxs=np.arange(0, 3), **cyc_kwargs),
    # r"Fix $R$, $F_1$ (AMJ)": fix_cycle(cycle_idxs=np.arange(3, 6), **cyc_kwargs),
    # r"Fix $R$, $F_1$ (JAS)": fix_cycle(cycle_idxs=np.arange(6, 9), **cyc_kwargs),
    # r"Fix $R$, $F_1$ (OND)": fix_cycle(cycle_idxs=np.arange(9, 12), **cyc_kwargs),
    # r"Fix $R$, $F_1$ (Jul-Dec)": fix_cycle(
    #     params=fits,
    #     pparams=fix_ddt_T(**pparam_kwargs),
    #     cycle_idxs=np.arange(6, 12),
    # ),
    # r"Fix $\varepsilon$": fix_eps(**pparam_kwargs),
    # r"Fix BJ": fits_fixed_BJ,
    # r"Fix $R_0$": fits_fixed_R0,
    # r"Fix Wyrtki": fits_fixed_wyrtki,
    # r"Fix $b$": get_perturbed_NRO(nro_form_idx=0, params=fits, nro_type="NROT_Lac"),
    # r"Fix $R_o$": fits_fixed_Ro,
    # r"Fix $\alpha$": fits_fixed_alpha,
    # r"Fix $\varepsilon$": fits_fixed_eps,
    # r"Fix Jan-Jun": fits_cyc0,
    # r"Fix Jul-Dec": fits_cyc1,°
    # r"Fixed $R_3$": src.utils.get_perturbed_multi(idxs=[(2, 2)], **pparam_kwargs),
    # r"Fix $L_o$": fits_fixed_Lo,
}

# ## get list of param sets and labels
param_sets = list(param_set_dict.values())
labels = list(param_set_dict.keys())

#### Plot parameter perturbations

Helper func to plot param set

In [ ]:
def plot_param_set(ax, params, model):
    """plot parameter set over time for given experiment"""

    ## get named named params (nnual mean)
    params_ = params.mean("cycle")

    ## plot core params
    for p in ["R", "epsilon", "F1", "F2", "bT_1", "cT_1", "bH_1"]:
        plot = ax.plot(params_.year, params_[p], label=p)
        ax.axhline(params_[p].isel(year=0), ls="--", c=plot[0].get_color(), lw=0.5)

    ## plot noise
    for p, c in zip(["xi_T", "xi_h"], ["gray", "lightgray"]):
        ax.plot(params_.year, params_[p], label=p, c=c)
        ax.axhline(params_[p].isel(year=0), ls="--", c=c, lw=0.5)

    ## format ax
    ax.set_xticks([])
    ax.legend(loc=(1.3, 0.1), prop=dict(size=6))

    return

Plot the param set

In [ ]:
fig, axs = plt.subplots(
    len(labels), 1, figsize=(4.5, 2.5 * len(labels)), layout="constrained"
)

## plot control
params0 = MODEL.get_RO_parameters(param_sets[0])
plot_param_set(axs[0], params0, model=MODEL)
axs[0].set_title(labels[0])

## plot diff from control
for ax, param_set, label in zip(axs[1:], param_sets[1:], labels[1:]):

    params_ = MODEL.get_RO_parameters(param_set)

    plot_param_set(ax, params_ - params0, model=MODEL)
    ax.set_title(label)

## formatting
src.utils.set_lims(axs[1:])
for ax in axs[1:]:
    ax.set_yticks([])


## legend
# axs[-1].legend(loc=(1.3, 0.1), prop=dict(size=8))
src.utils.add_vticks(axs, xticks=[], xlines=[2010])

plt.show()

### Run simulations

In [ ]:
exp_kwargs = dict(**simulation_kwargs, model=MODEL)
sims_exp = [sims] + [get_sims_over_time(params=p, **exp_kwargs) for p in param_sets[1:]]

Plot some samples

### Compute stats from simulations

In [ ]:
## stat
get_q = lambda x: x.quantile(q=[0.1, 0.5, 0.9], dim="member").rename({"quantile": "q"})

## funcs to get stats over time
# season = xr.groupers.SeasonGrouper(["JFM", "AMJ", "JAS", "OND"])
season = xr.groupers.SeasonGrouper(["DJF", "MAM", "JJA", "SON"])
get_fn_ot = lambda exps, fn: [get_q(exp.groupby(time=season).map(fn)) for exp in exps]
get_fn_ot2 = lambda exps, fn: [exp.groupby(time=season).map(fn) for exp in exps]

## compute
sigma_exp = get_fn_ot(sims_exp, lambda x: x.std("time"))
quantiles_exp = get_fn_ot2(sims_exp, get_stats)
var_exp = get_fn_ot2(sims_exp, var_by_quant)

## SPECIFY PLOT SEASON
PLOT_SEASON = "DJF"

### Plot results

#### Absolute stats from each experiments

##### Variance

In [ ]:
def format_row(axs, y0, season):
    """format row of plot in comparison"""

    ## label
    axs[0].set_yticks([np.round(sigma0, 1), np.round(sigma0 + 0.5, 1)])
    for ax in axs[1:]:
        ax.set_ylabel(None)
    axs[-1].set_ylabel(season)
    axs[-1].yaxis.set_label_position("right")
    for ax in axs:
        ax.axhline(y0, c="gray", lw=1, ls="--")
        ax.set_ylim([y0 - 0.3, y0 + 0.7])

    return


def format_subplots(axs, labels):
    """format all subplots"""
    for ax in axs[:-1].flatten():
        ax.set_xticks([])
        ax.set_xlabel(None)

    for ax in axs[:, 1:].flatten():
        ax.set_yticks([])

    for j in range(axs.shape[1]):
        axs[0, j].set_title(labels[j])

    return


def plot_stats_comp(ax, list_of_stats, labels, colors=None, n=varnames[0]):
    """plot comparison of variance over time"""

    if colors is None:
        colors = sns.color_palette()[: len(list_of_stats)]

    for stats, label, c in zip(list_of_stats, labels, colors):

        ## plot median
        mplot = ax.plot(stats.year, stats[n].sel(q=0.5), lw=2.5, label=label, c=c)

        ## plot lower/upper quantiles
        kwargs = dict(c=mplot[0].get_color(), lw=0.8)
        for q in stats.q:
            if q != 0.5:
                ax.plot(stats.year, stats[n].sel(q=q), **kwargs)

    ## label and set plotting specs
    ax.set_xlabel("Year")
    ax.set_ylabel(r"$\sigma_T$ ($^{\circ}$C)")
    ax.set_ylim([0.3, 1.7])
    ax.set_xticks([1870, 2010, 2080])
    ax.set_yticks([0.6, 1.2])

    return

In [ ]:
## specify which variable to plot
PLOT_VAR = varnames[0]

## plot results
n = len(sigma_exp) - 1
fig, axs = plt.subplots(4, n, figsize=(n * 2, 7), layout="constrained")

for i, s in enumerate(sigma_exp[0].season.values):

    ## loop thru experiments
    for j in np.arange(len(sigma_exp) - 1):
        plot_stats_comp(
            axs[i, j],
            [sigma_exp[0].sel(season=s), sigma_exp[j + 1].sel(season=s)],
            labels=[labels[0], labels[j + 1]],
            colors=["k", sns.color_palette()[np.mod(j, 10)]],
            n=PLOT_VAR,
        )

    ## get baseline value
    sigma0 = sigma_exp[0].sel(season=s)[PLOT_VAR].isel(q=1, year=0).values.item()

    ## format
    format_row(axs=axs[i], y0=sigma0, season=s)

## format all subplots
format_subplots(axs, labels=labels[1:])
src.utils.add_vticks(axs.flatten(), xticks=[1870, 2010], xlines=[2010])

# save(fig, fname="ro-experiments")

plt.show()

##### Quantiles

In [ ]:
## SPECIFY SEASON
SEASON = PLOT_SEASON

fig, axs = plt.subplots(
    1, len(quantiles_exp), figsize=(3 * len(quantiles_exp), 3), layout="constrained"
)
for ax, stats_, title in zip(axs, quantiles_exp, labels):
    q = stats_.sel(season=SEASON)["q"].sel(v=PLOT_VAR)

    ax.plot(stats_.year, q.sel(quantile=0.95), c="r", label="El Niño")
    ax.plot(stats_.year, -q.sel(quantile=0.05), c="b", label="La Niña")
    ax.plot(
        stats_.year,
        0.5 * (q.sel(quantile=0.95) - q.sel(quantile=0.05)),
        label="Average",
        c="k",
    )

    ax_kwargs = dict(ls="--", c="gray", lw=0.8)
    add_vticks(axs, xticks=[1870, 2010, 2050], xlines=[2010, 2050])
    ax.set_title(title)

src.utils.set_lims(axs[:-1])
for ax in axs[1:]:
    axs[1].set_yticks([])
axs[0].legend()
axs[-1].set_ylim(axs[0].get_ylim())

## save to file
save(fig, fname="RO-experiments", do_save=False)

plt.show()

#### Plot effect

Compute effect

In [ ]:
## for quantiles
ctrl_ = quantiles_exp[0]
quantiles_eff = [ctrl_ - q for q in quantiles_exp]

## for variance
ctrl_var = var_exp[0]
var_eff = [ctrl_var - q for q in var_exp]

Plotting function

In [ ]:
def plot_quantiles(ax, stats, season, plot_var, lev=0.95):
    """plot quantiles on ax"""

    ## extract quantile data for plotting
    q = stats.sel(season=season)["q"].sel(v=plot_var)

    ## get el nino/la nina quantiles
    q_warm = q.sel(quantile=lev, method="nearest")
    q_cold = -q.sel(quantile=(1 - lev), method="nearest")

    ## plot data
    ax.plot(stats.year, q_warm, c="r", label="El Niño")
    ax.plot(stats.year, q_cold, c="b", label="La Niña")

    ## plot mean
    ax.plot(stats.year, 0.5 * (q_warm + q_cold), label="Average", c="k")

    ## formatting
    src.utils.add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010])
    ax.axhline(0, ls="--", c="k", lw=0.8)
    ax.set_yticks([])

    return

##### Quantiles

Effect of each experiment

In [ ]:
## specify seaons
SEASON = PLOT_SEASON

## number of figures in panel
n = len(quantiles_exp) - 1

## get control
ctrl = ctrl_.sel(season=SEASON)["q"].sel(v=PLOT_VAR)

## make plot
fig, axs = plt.subplots(1, n, figsize=(3 * (n), 3), layout="constrained")
for ax, stats_, title in zip(axs, quantiles_eff[1:], labels[1:]):

    plot_quantiles(ax, stats=stats_, season=SEASON, plot_var=PLOT_VAR)

    ax.set_title(f"Effect of {title[4:]}")
    # ax.set_ylim([-0.3, 0.4])
    # ax.set_ylim([-0.05, 0.2])

src.utils.set_lims(axs)
axs[0].set_yticks([-0.8, -0.4, 0, 0.4])

axs[-1].set_ylim(axs[0].get_ylim())
axs[0].legend()

plt.show()

Check linearity

In [ ]:
SEASON = PLOT_SEASON

## specify indices
idxs = [1, 5, 6, 7]

## compute number of panels for plot
n = len(idxs) + 2

## get plot vars and labels
plot_vars = [quantiles_eff[i] for i in idxs]
plot_labs = [labels[i] for i in idxs]

## get control effect
ctrl_eff = quantiles_exp[0] - quantiles_exp[0].isel(year=0)

## get sum
sum_ = xr.zeros_like(quantiles_eff[0])
for i in idxs:
    sum_ += quantiles_eff[i]

fig, axs = plt.subplots(1, n, figsize=(2 * n, 2), layout="constrained")

## plot experiments
for ax, stats_, title in zip(axs, plot_vars, plot_labs):

    ## plot data
    plot_quantiles(ax, stats_, season=SEASON, plot_var=PLOT_VAR)
    ax.set_title(f"Effect of {title[4:]}")

## plot sum
plot_quantiles(axs[-2], sum_, season=SEASON, plot_var=PLOT_VAR)
axs[-2].set_title("Sum")

## plo tcontrol
plot_quantiles(axs[-1], ctrl_eff, season=SEASON, plot_var=PLOT_VAR)
axs[-1].set_title("Control (all params)")

## formatting
src.utils.set_lims(axs)
axs[0].set_yticks([-0.4, 0, 0.4])
axs[0].set_ylabel(r"$^{\circ}$C")
axs[0].legend(prop=dict(size=8), loc="lower left")

## save
save(fig, fname="exp-comparison_general", do_save=False)

plt.show()

Check linearity of subset

In [ ]:
SEASON = PLOT_SEASON

## specify indices
idxs = [4, -4]

## get control index
ctrl_idx = 5

## compute number of panels for plot
n = len(idxs) + 2

## get plot vars and labels
plot_vars = [quantiles_eff[i] for i in idxs]
plot_labs = [labels[i] for i in idxs]

## get sum
sum_ = xr.zeros_like(quantiles_eff[0])
for i in idxs:
    sum_ += quantiles_eff[i]

fig, axs = plt.subplots(1, n, figsize=(2.5 * n, 2.5), layout="constrained")

## plot experiments
for ax, stats_, title in zip(axs, plot_vars, plot_labs):

    ## plot data
    plot_quantiles(ax, stats_, season=SEASON, plot_var=PLOT_VAR)
    ax.set_title(f"Effect of {title[4:]}")

## plot sum
plot_quantiles(axs[-2], sum_, season=SEASON, plot_var=PLOT_VAR)
axs[-2].set_title("Sum")

## plo tcontrol
plot_quantiles(axs[-1], quantiles_eff[ctrl_idx], season=SEASON, plot_var=PLOT_VAR)
axs[-1].set_title("Control (effect of $R$)")

## formatting
src.utils.set_lims(axs)
axs[0].set_yticks([-0.8, -0.4, 0, 0.4])
axs[0].set_ylabel(r"$^{\circ}$C")
axs[0].legend()

## save to file
save(fig, fname="exp-comparison_Ro", do_save=False)

plt.show()

In [ ]:
SEASON = PLOT_SEASON

## specify indices
idxs = [-3, -2, -1]

## get control index
ctrl_idx = -4

## compute number of panels for plot
n = len(idxs) + 2

## get plot vars and labels
plot_vars = [quantiles_eff[i] for i in idxs]
plot_labs = [labels[i] for i in idxs]

## get sum
sum_ = xr.zeros_like(quantiles_eff[0])
for i in idxs:
    sum_ += quantiles_eff[i]

fig, axs = plt.subplots(1, n, figsize=(2.5 * n, 2.5), layout="constrained")

## plot experiments
for ax, stats_, title in zip(axs, plot_vars, plot_labs):

    ## plot data
    plot_quantiles(ax, stats_, season=SEASON, plot_var=PLOT_VAR)
    ax.set_title(f"Effect of {title[4:]}")

## plot sum
plot_quantiles(axs[-2], sum_, season=SEASON, plot_var=PLOT_VAR)
axs[-2].set_title("Sum")

## plo tcontrol
plot_quantiles(axs[-1], quantiles_eff[ctrl_idx], season=SEASON, plot_var=PLOT_VAR)
axs[-1].set_title(r"Control (effect of $\alpha$)")

## formatting
src.utils.set_lims(axs)
axs[0].set_yticks([-0.8, -0.4, 0, 0.4])
axs[0].set_ylabel(r"$^{\circ}$C")
axs[0].legend()


## save to file
save(fig, fname="exp-comparison_alpha", do_save=False)

plt.show()

##### Variance

In [ ]:
SEASON = PLOT_SEASON

idxs = [1, 3, 4]
# idxs = [1,2]
n = len(idxs) + 2
plot_vars = [var_eff[i] for i in idxs]
plot_labs = [labels[i] for i in idxs]

## get control effect
ctrl_eff = var_exp[0] - var_exp[0].isel(year=0)

## get sum
sum_ = xr.zeros_like(var_eff[0])
for i in idxs:
    sum_ += var_eff[i]

fig, axs = plt.subplots(1, n, figsize=(2.5 * n, 2.5), layout="constrained")
for ax, stats_, title in zip(
    axs, plot_vars + [sum_] + [ctrl_eff], plot_labs + ["sum"] + ["Control"]
):
    q = stats_[varnames[0]].sel(season=SEASON)

    ax.plot(
        stats_.year,
        q.isel(intensity=2),
        c="r",
        label="El Niño",
    )
    ax.plot(
        stats_.year,
        q.isel(intensity=0),
        c="b",
        label="La Niña",
    )
    ax.plot(
        stats_.year,
        0.5 * q.sum("intensity"),
        label="Average",
        c="k",
    )

    ax_kwargs = dict(ls="--", c="gray", lw=0.8)
    add_vticks(axs, xticks=[1870, 2010, 2050], xlines=[2010, 2050])
    ax.axhline(0, ls="--", c="k", lw=0.8)
    ax.set_title(f"Effect of {title[4:]}")
    # ax.set_ylim([-0.2, 0.28])

axs[-2].set_title("Sum")
axs[-1].set_title("Control")
src.utils.set_lims(axs)
for ax in axs[1:]:
    axs[1].set_yticks([])
axs[0].legend()

plt.show()